# Getting started with TinyTimeMixer (TTM)

This notebooke demonstrates the usage of a pre-trained `TinyTimeMixer` model for several multivariate time series forecasting tasks. For details related to model architecture, refer to the [TTM paper](https://arxiv.org/pdf/2401.03955.pdf).

In this example, we will use a pre-trained TTM-512-96 model. That means the TTM model can take an input of 512 time points (`context_length`), and can forecast upto 96 time points (`forecast_length`) in the future. We will use the pre-trained TTM in two settings:
1. **Zero-shot**: The pre-trained TTM will be directly used to evaluate on the `test` split of the target data. Note that the TTM was NOT pre-trained on the target data.
2. **Few-shot**: The pre-trained TTM will be quickly fine-tuned on only 5% of the `train` split of the target data, and subsequently, evaluated on the `test` part of the target data.

Note: Alternatively, this notebook can be modified to try any other TTM model from a suite of TTM models. For details, visit the [Hugging Face TTM Model Repository](https://huggingface.co/ibm-granite/granite-timeseries-ttm-r2).

1. IBM Granite TTM-R1 pre-trained models can be found here: [Granite-TTM-R1 Model Card](https://huggingface.co/ibm-granite/granite-timeseries-ttm-r1)
2. IBM Granite TTM-R2 pre-trained models can be found here: [Granite-TTM-R2 Model Card](https://huggingface.co/ibm-granite/granite-timeseries-ttm-r2)
3. Research-use (non-commercial use only) TTM-R2 pre-trained models can be found here: [Research-Use-TTM-R2](https://huggingface.co/ibm-research/ttm-research-r2)

### The get_model() utility
TTM Model card offers a suite of models with varying `context_length` and `prediction_length` combinations.
In this notebook, we will utilize the TSFM `get_model()` utility that automatically selects the right model based on the given input `context_length` and `prediction_length` (and some other optional arguments) abstracting away the internal complexity. See the usage examples below in the `zeroshot_eval()` and `fewshot_finetune_eval()` functions. For more details see the [docstring](https://github.com/ibm-granite/granite-tsfm/blob/main/tsfm_public/toolkit/get_model.py) of the function definition.

## Install `tsfm` 
**[Optional for Local Run / Mandatory for Google Colab]**  
Run the below cell to install `tsfm`. Skip if already installed.

In [1]:
# # Install the tsfm library
# ! pip install "granite-tsfm[notebooks] @ git+https://github.com/ibm-granite/granite-tsfm.git@v0.3.3"

## Imports

In [2]:
import math
import os
import tempfile

import pandas as pd
import numpy as np
from torch.optim import AdamW
from torch.optim.lr_scheduler import OneCycleLR
from transformers import EarlyStoppingCallback, Trainer, TrainingArguments, set_seed
from transformers.integrations import INTEGRATION_TO_CALLBACK

from tsfm_public import TimeSeriesPreprocessor, TrackingCallback, count_parameters, get_datasets
from tsfm_public.toolkit.get_model import get_model
from tsfm_public.toolkit.lr_finder import optimal_lr_finder
from tsfm_public.toolkit.visualization import plot_predictions
import warnings


# Suppress all warnings
warnings.filterwarnings("ignore")

## Zero-shot evaluation method

In [3]:
def zeroshot_eval(dataset_name, batch_size, data, context_length=512, forecast_length=12, ):
    # Get data

    tsp = TimeSeriesPreprocessor(
        **column_specifiers,
        context_length=context_length,
        prediction_length=forecast_length,
        scaling=True,
        encode_categorical=False,
        scaler_type="standard",
    )

    # Load model
    zeroshot_model = get_model(
        TTM_MODEL_PATH,
        context_length=context_length,
        prediction_length=forecast_length,
        freq_prefix_tuning=False,
        freq=None,
        prefer_l1_loss=False,
        prefer_longer_context=True,
    )

    dset_train, dset_valid, dset_test = get_datasets(
        tsp, data, split_config, use_frequency_token=zeroshot_model.config.resolution_prefix_tuning
    )
    temp_dir = tempfile.mkdtemp()
    zeroshot_trainer = Trainer(
        model=zeroshot_model,
        args=TrainingArguments(
            output_dir=temp_dir,
            per_device_eval_batch_size=batch_size,
            seed=SEED,
            report_to="none",
        ),
    )

    # predict() runs inference once and returns both predictions and metrics
    predictions_dict = zeroshot_trainer.predict(dset_train)
    predictions_np = predictions_dict.predictions[0]
    
    return dset_train, predictions_np, tsp


In [4]:
import json
from datetime import datetime
from tqdm import tqdm
SEED = 42
set_seed(SEED)

# TTM Model path. The default model path is Granite-R2. Below, you can choose other TTM releases.
TTM_MODEL_PATH = "ibm-granite/granite-timeseries-ttm-r2"
# TTM_MODEL_PATH = "ibm-granite/granite-timeseries-ttm-r1"
# TTM_MODEL_PATH = "ibm-research/ttm-research-r2"

# Context length, Or Length of the history.
# Currently supported values are: 512/1024/1536 for Granite-TTM-R2 and Research-Use-TTM-R2, and 512/1024 for Granite-TTM-R1
CONTEXT_LENGTH = 512
#1  week or 2 weeks, predict for next 2 days
# Granite-TTM-R2 supports forecast length upto 720 and Granite-TTM-R1 supports forecast length upto 96
# Arima? Rolling average, Rolling median 
PREDICTION_LENGTH = 12
OUT_DIR = "ttm_finetuned_models/"

In [5]:
timestamp_column = "Timestamp"
id_columns = []  # mention the ids that uniquely identify a time-series.

target_columns = [
 'PM2.5 (µg/m³)',
 'PM10 (µg/m³)',
 'NO2 (µg/m³)',
 'SO2 (µg/m³)',
 'CO (mg/m³)',
 'Ozone (µg/m³)',
]


split_config = {
    "train": 1.0,
    "test": 0.0,
}

column_specifiers = {
    "timestamp_column": timestamp_column,
    "id_columns": id_columns,
    "target_columns": target_columns,
    "control_columns": [],
}


In [6]:
# d = pd.read_csv(r"/home/student/rishi/sites_imputed/site_113_Shadipur_Delhi_CPCB_15Min.csv", parse_dates=[timestamp_column])
# dset_train, preds = zeroshot_eval(
#             dataset_name='site_name',
#             data=d,
#             context_length=CONTEXT_LENGTH,
#             forecast_length=PREDICTION_LENGTH,
#             batch_size=64
#         )


In [7]:
# import pickle as pkl
# df=pd.read_pickle(r'/home/student/rishi/ttm_preds_v1.pkl')

In [8]:
# # --- Quick test on a single site ---
# import torch
# import pickle as pkl

# test_folder = r"/home/student/rishi/sites_imputed"
# test_results_root = r"/home/student/rishi/ttm_results_v1"
# os.makedirs(test_results_root, exist_ok=True)

# # Pick one file to test
# test_file = sorted(f for f in os.listdir(test_folder) if os.path.isfile(os.path.join(test_folder, f)))[0]
# print(f"Testing with: {test_file}")

# site_name = os.path.splitext(test_file)[0]
# site_dir = os.path.join(test_results_root, site_name)
# # os.makedirs(site_dir, exist_ok=True)

# df = pd.read_csv(os.path.join(test_folder, test_file), parse_dates=[timestamp_column])
# print(f"Data shape: {df.shape}")

# dset_train, preds, tsp = zeroshot_eval(
#     dataset_name='site_name',
#     data=df,
#     context_length=CONTEXT_LENGTH,
#     forecast_length=PREDICTION_LENGTH,
#     batch_size=64,
# )

# # --- 1) Save dataset as torch tensor ---
# past_list = []
# future_list = []
# timestamps=[]
# for i in range(len(dset_train)):
#     item = dset_train[i]
#     past_list.append(item['past_values'])
#     future_list.append(item['future_values'])
#     timestamps.append(dset_train[i]['timestamp'])
# past_tensor = torch.stack(past_list)
# future_tensor = torch.stack(future_list)
# # torch.save({"past_values": past_tensor, "future_values": future_tensor, "timestamps": timestamps},
#         #    os.path.join(site_dir, "dataset.pt"))
# print(f"Dataset: past {past_tensor.shape}, future {future_tensor.shape}")

# # --- 2) Save predictions as torch tensor ---
# preds_tensor = torch.tensor(preds, dtype=torch.float32)
# # torch.save(preds_tensor, os.path.join(site_dir, "predictions.pt"))
# print(f"Predictions tensor shape: {preds_tensor.shape}")

# # --- 3) Save scaler params pickle (all you need for inverse scaling) ---
# scaler_key = "0"
# scaler_obj = tsp.target_scaler_dict[scaler_key]
# scaler_params = {
#     "mean_": scaler_obj.mean_.tolist(),
#     "scale_": scaler_obj.scale_.tolist(),
#     "target_columns": target_columns,
#     "scaler_type": "standard",
# }
# # with open(os.path.join(site_dir, "scaler_params.pkl"), "wb") as f:
# #     pkl.dump(scaler_params, f)

# print(f"Mean: {scaler_params['mean_']}")
# print(f"Scale: {scaler_params['scale_']}")

# # print(f"\nSaved files: {os.listdir(site_dir)}")


In [9]:
# print("scaling_id_columns:", tsp.scaling_id_columns)
# print("target_scaler_dict keys:", list(tsp.target_scaler_dict.keys()))

# # Print mean/scale matched to each target column
# print("\n{:<30} {:>15} {:>15}".format("Column", "Mean", "Scale"))
# print("-" * 62)
# for col, m, s in zip(target_columns, tsp.target_scaler_dict["0"].mean_, tsp.target_scaler_dict["0"].scale_):
#     print(f"{col:<30} {m:>15.4f} {s:>15.4f}")

# # Cross-check: manually scale a known row from the original df and compare
# row = df[target_columns].iloc[0].values
# print("\nOriginal row:", row)

# scaled_manual = (row - tsp.target_scaler_dict["0"].mean_) / tsp.target_scaler_dict["0"].scale_
# scaled_tsp = tsp.target_scaler_dict["0"].transform(df[target_columns].iloc[[0]].values)
# print("Manual scaled:", scaled_manual)
# print("TSP scaled:   ", scaled_tsp[0])

# # Manual inverse: reconstruct original from scaled
# inv_manual = scaled_manual * tsp.target_scaler_dict["0"].scale_ + tsp.target_scaler_dict["0"].mean_
# inv_tsp = tsp.target_scaler_dict["0"].inverse_transform(scaled_manual.reshape(1, -1))
# print("\nManual inverse:", inv_manual)
# print("TSP inverse:   ", inv_tsp[0])
# print("Match original:", np.allclose(inv_manual, row))

In [10]:
import torch
import pickle as pkl

folder = r"/home/student/rishi/sites_imputed"
results_root = r"/home/student/rishi/ttm_results_v1"
os.makedirs(results_root, exist_ok=True)

files = [f for f in sorted(os.listdir(folder)) if os.path.isfile(os.path.join(folder, f))]

for file in tqdm(files, desc="Processing sites"):
    site_name = os.path.splitext(file)[0]
    site_dir = os.path.join(results_root, site_name)
    os.makedirs(site_dir, exist_ok=True)

    df = pd.read_csv(os.path.join(folder, file), parse_dates=[timestamp_column])
    dset_train, preds, tsp = zeroshot_eval(
        dataset_name='site_name',
        data=df,
        context_length=CONTEXT_LENGTH,
        forecast_length=PREDICTION_LENGTH,
        batch_size=64,
    )

    # --- 1) Save dataset as torch tensor ---
    past_list, future_list, timestamps = [], [], []
    for i in range(len(dset_train)):
        item = dset_train[i]
        past_list.append(item['past_values'])
        future_list.append(item['future_values'])
        timestamps.append(dset_train[i]['timestamp'])
    past_tensor = torch.stack(past_list)
    future_tensor = torch.stack(future_list)
    torch.save({"past_values": past_tensor, "future_values": future_tensor, "timestamps": timestamps},
               os.path.join(site_dir, "dataset.pt"))

    # --- 2) Save predictions as torch tensor ---
    preds_tensor = torch.tensor(preds, dtype=torch.float32)
    torch.save(preds_tensor, os.path.join(site_dir, "predictions.pt"))

    # --- 3) Save scaler params for inverse scaling ---
    scaler_key = "0"
    scaler_obj = tsp.target_scaler_dict[scaler_key]
    scaler_params = {
        "mean_": scaler_obj.mean_.tolist(),
        "scale_": scaler_obj.scale_.tolist(),
        "target_columns": target_columns,
        "scaler_type": "standard",
    }
    with open(os.path.join(site_dir, "scaler_params.pkl"), "wb") as f:
        pkl.dump(scaler_params, f)

    print(f"Saved {site_name}: past {past_tensor.shape}, future {future_tensor.shape}, preds {preds_tensor.shape}")

print(f"\nAll results saved to {results_root}")


Processing sites:   0%|          | 0/138 [00:00<?, ?it/s]INFO:p-317725:t-140541437495104:get_model.py:get_model:Loading model from: ibm-granite/granite-timeseries-ttm-r2
INFO:p-317725:t-140541437495104:get_model.py:get_model:Model loaded successfully from ibm-granite/granite-timeseries-ttm-r2, revision = 512-48-ft-r2.1.
INFO:p-317725:t-140541437495104:get_model.py:get_model:[TTM] context_length = 512, prediction_length = 48


Processing sites:   1%|          | 1/138 [00:14<32:20, 14.17s/it]INFO:p-317725:t-140541437495104:get_model.py:get_model:Loading model from: ibm-granite/granite-timeseries-ttm-r2


Saved site_113_Shadipur_Delhi_CPCB_15Min: past torch.Size([25781, 512, 6]), future torch.Size([25781, 12, 6]), preds torch.Size([25781, 12, 6])


INFO:p-317725:t-140541437495104:get_model.py:get_model:Model loaded successfully from ibm-granite/granite-timeseries-ttm-r2, revision = 512-48-ft-r2.1.
INFO:p-317725:t-140541437495104:get_model.py:get_model:[TTM] context_length = 512, prediction_length = 48


Processing sites:   1%|▏         | 2/138 [00:40<48:18, 21.31s/it]INFO:p-317725:t-140541437495104:get_model.py:get_model:Loading model from: ibm-granite/granite-timeseries-ttm-r2


...

Processing sites:  99%|█████████▉| 137/138 [36:06<00:17, 17.08s/it]INFO:p-317725:t-140541437495104:get_model.py:get_model:Loading model from: ibm-granite/granite-timeseries-ttm-r2


Saved site_5675_Raghunathpali_Rourkela_OSPCB_15Min: past torch.Size([25781, 512, 6]), future torch.Size([25781, 12, 6]), preds torch.Size([25781, 12, 6])


INFO:p-317725:t-140541437495104:get_model.py:get_model:Model loaded successfully from ibm-granite/granite-timeseries-ttm-r2, revision = 512-48-ft-r2.1.
INFO:p-317725:t-140541437495104:get_model.py:get_model:[TTM] context_length = 512, prediction_length = 48


Processing sites: 100%|██████████| 138/138 [36:22<00:00, 15.82s/it]

Saved site_5680_Lingaraj_Nagar_Hubballi_KSPCB_15Min: past torch.Size([25781, 512, 6]), future torch.Size([25781, 12, 6]), preds torch.Size([25781, 12, 6])

All results saved to /home/student/rishi/ttm_results_v1
